# Model Watch: see Gemma-2 2B choose each word

Run this top to bottom. You will watch the model write, one word at a time, and for every word see:

- **The choice**: the word picked and its runners-up.
- **How the guess formed**: what each of the 26 layers would have said if the model stopped there. You see the answer firm up.
- **Active concepts**: which features fired at layer 20, with labels from Neuronpedia.

Nothing in the model is changed. This only observes.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (free).

## 1. Get the project files

Either paste your GitHub URL for the project into `REPO_URL`, or leave it empty and upload `model-watch.zip` when asked.

In [ ]:
import os
REPO_URL = ""  # e.g. "https://github.com/<you>/model-watch"

if not os.path.exists("model-watch"):
    if REPO_URL:
        !git clone -q {REPO_URL} model-watch
    else:
        from google.colab import files
        print("Choose model-watch.zip")
        uploaded = files.upload()
        zip_name = next(iter(uploaded))
        !unzip -q -o "{zip_name}"
%cd model-watch
!pip install -q -r requirements.txt

## 2. Unlock Gemma on Hugging Face (one time)

1. Sign in at huggingface.co and accept the license on https://huggingface.co/google/gemma-2-2b. The Gemma Scope SAEs at https://huggingface.co/google/gemma-scope-2b-pt-res may ask too.
2. Create a read token at https://huggingface.co/settings/tokens.
3. In Colab, click the key icon in the left sidebar, add a secret named `HF_TOKEN`, and turn on notebook access. If you skip this, the next cell shows a login box instead.

In [ ]:
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("Using HF_TOKEN from Colab secrets.")
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()

## 3. Load the model and the concept dictionary

Several GB of model weights and about 300 MB of SAE weights download the first time. Takes a few minutes.

In [ ]:
import torch
from model_watch import ModelWatcher, WatchConfig, step_html, save_json, export_html, trace_to_html

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be slow)")
watcher = ModelWatcher.load(WatchConfig(), hf_token=os.environ.get("HF_TOKEN"))

## 4. Watch it write

The panel below updates live as each word is chosen. Change `PROMPT`, `TOKENS` or `DELAY` and run the cell again.

In [ ]:
import time
from IPython.display import display, HTML

PROMPT = "The capital of France is"
TOKENS = 15
DELAY = 0.8  # seconds to pause on each word so you can read it

panel = display(HTML("<i>Thinking...</i>"), display_id=True)
text = PROMPT

def show(record, steps):
    global text
    panel.update(HTML(step_html(record, text, watcher.sae_layer, TOKENS)))
    text += record["token"]
    time.sleep(DELAY)

trace = watcher.trace(PROMPT, TOKENS, on_step=show)
print("Final text:", text)

## 5. Replay it

The replay viewer lets you step back and forth, click any word, and see the full layer table. It is saved as `trace.html`, a single file that opens in any browser.

In [ ]:
import html
save_json(trace, "trace.json")
export_html(trace, "trace.html")
display(HTML(f'<iframe srcdoc="{html.escape(trace_to_html(trace))}" style="width:100%;height:1000px;border:0"></iframe>'))

In [ ]:
# Download the replay page and raw data to your computer
from google.colab import files
files.download("trace.html")
files.download("trace.json")

## How to read it

- **Settled at layer N** is the first layer from which the top guess equals the final word and never changes again. Facts often settle in the middle-to-late layers. Grammar words like "the" or "." can settle early or bounce around until the end.
- **Early layers look odd.** The logit lens reads every layer through the model's final output step. Early layers were not trained to be read that way, so their guesses are rough. Watch the trend, not the first few rows.
- **Labels are a first check, not proof.** Neuronpedia labels are written by an AI from examples where each feature fires. If a label doesn't fit what you see, click through to the feature page.
- **First-run check:** on "The capital of France is", the layer-20 concepts should look geographic or factual. If the labels look random, the dictionary and the labels may not match. Try `WatchConfig(sae_l0=...)` with another value listed in the Gemma Scope repo (22, 38, 71, 139, 294 for layer 20, width 16k).

## Prompts to try

- `"The Eiffel Tower is located in the city of"` (a fact the model knows)
- `"def is_even(n):\n    return"` (code)
- `"Q: What is 17 + 25?\nA:"` (arithmetic: watch when the number settles)
- `"The doctor told the nurse that"` (watch which pronoun it picks and when)
- `"The Marine Corps was founded on November 10,"` (a date)